In [ ]:
from openai import OpenAI
import pandas as pd
import json
import time
from pathlib import Path

# -----------------------------
# LLM Client Setup
# -----------------------------
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
model_name = "qwen/qwen3-vl-30b"

# -----------------------------
# File Paths
# -----------------------------
input_candidates = [
    Path("2026-05-15_Price_Data_USD.xls"),
    Path("price_info/2026-05-15_Price_Data_USD.xls"),
]

INPUT_FILE = next((p for p in input_candidates if p.exists()), input_candidates[0])
OUTPUT_FILE = INPUT_FILE.with_name("2026-05-15_Price_Data_USD_lmstudio_extracted.csv")

# -----------------------------
# Target columns to extract from main_content
# -----------------------------
TARGET_COLUMNS = [
    "Amount Paid by Buyer",
    "Amount advertised by seller",
    "Amount Promised to Seller",
    "Amount Paid to Seller",
    "Fee Charged by Broker",
    "Fee received by Surgeon",
    "Currency used",
    "Correct (1/0)",
    "currency",
    "Cleaned currency code/label for USD conversion",
    "Country/area used to merge World Bank exchange rate",
    "Duplicate",
    "country",
    "Remarks",
    "multiple_sellers",
    "multiple_buyers",
    "multiple_brokers",
    "multiple_surgery",
    "Clean numeric AmountPaidbyBuyer (first amount, local currency)",
    "Minimum AmountPaidbyBuyer (local currency)",
    "Maximum AmountPaidbyBuyer (local currency)",
    "Average AmountPaidbyBuyer (local currency)",
    "Number of amounts parsed from AmountPaidbyBuyer",
    "Clean numeric AmountPromisedtoSeller (first amount, local currency)",
    "Minimum AmountPromisedtoSeller (local currency)",
    "Maximum AmountPromisedtoSeller (local currency)",
    "Average AmountPromisedtoSeller (local currency)",
    "Number of amounts parsed from AmountPromisedtoSeller",
    "Clean numeric AmountPaidtoSeller (first amount, local currency)",
    "Minimum AmountPaidtoSeller (local currency)",
    "Maximum AmountPaidtoSeller (local currency)",
    "Average AmountPaidtoSeller (local currency)",
    "Number of amounts parsed from AmountPaidtoSeller",
    "Clean numeric FeeChargedbyBroker (first amount, local currency)",
    "Minimum FeeChargedbyBroker (local currency)",
    "Maximum FeeChargedbyBroker (local currency)",
    "Average FeeChargedbyBroker (local currency)",
    "Number of amounts parsed from FeeChargedbyBroker",
    "Clean numeric FeereceivedbySurgeon (first amount, local currency)",
    "Minimum FeereceivedbySurgeon (local currency)",
    "Maximum FeereceivedbySurgeon (local currency)",
    "Average FeereceivedbySurgeon (local currency)",
    "Number of amounts parsed from FeereceivedbySurgeon",
    "Exchange rate, local currency units per USD",
    "AmountPaidbyBuyer_USD",
    "AmountPaidbyBuyer_min_USD",
    "AmountPaidbyBuyer_max_USD",
    "AmountPaidbyBuyer_avg_USD",
    "AmountPromisedtoSeller_USD",
    "AmountPromisedtoSeller_min_USD",
    "AmountPromisedtoSeller_max_USD",
    "AmountPromisedtoSeller_avg_USD",
    "AmountPaidtoSeller_USD",
    "AmountPaidtoSeller_min_USD",
    "AmountPaidtoSeller_max_USD",
    "AmountPaidtoSeller_avg_USD",
    "FeeChargedbyBroker_USD",
    "FeeChargedbyBroker_min_USD",
    "FeeChargedbyBroker_max_USD",
    "FeeChargedbyBroker_avg_USD",
    "FeereceivedbySurgeon_USD",
    "FeereceivedbySurgeon_min_USD",
    "FeereceivedbySurgeon_max_USD",
    "FeereceivedbySurgeon_avg_USD",
]

# -----------------------------
# JSON schema for extraction
# -----------------------------
price_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "PriceInfoExtraction",
        "schema": {
            "type": "object",
            "properties": {
                col: {
                    "type": "string",
                    "description": f"Extracted value for: {col}. Return empty string if not available.",
                }
                for col in TARGET_COLUMNS
            },
            "required": TARGET_COLUMNS,
            "additionalProperties": False,
        },
    },
}

# -----------------------------
# Load input XLS
# -----------------------------
df = pd.read_excel(INPUT_FILE, dtype=str).fillna("")

if "main_content" not in df.columns:
    raise ValueError("Column 'main_content' not found in the input file.")

# Keep original data intact in memory; fill extracted columns in output dataframe
out_df = df.copy()
for col in TARGET_COLUMNS:
    out_df[col] = ""

# -----------------------------
# Loop through rows and extract
# -----------------------------
for i, row in out_df.iterrows():
    start_time = time.time()
    content = str(row.get("main_content", "")).strip()

    if not content:
        print(f"Row {i + 1}: main_content is empty, skipped.")
        continue

    prompt_text = f"""
You extract structured price-related information from kidney trade news text.

Task:
- Read the article text.
- Populate every field in the schema.
- Return JSON only.

Rules:
- If a field is missing or unclear, return an empty string for that field.
- For monetary fields, keep the amount exactly as stated when possible.
- For numeric summary fields, return plain numeric strings (no extra words).
- For 0/1 flags, return "0" or "1" when inferable, otherwise empty string.
- Do not invent facts.

Article:
{content}
""".strip()

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": [{"type": "text", "text": prompt_text}],
                }
            ],
            response_format=price_schema,
        )

        raw = response.choices[0].message.content

        if isinstance(raw, str):
            cleaned = raw.strip()
            if cleaned.startswith("```"):
                cleaned = cleaned.strip("`")
                cleaned = cleaned.replace("json", "", 1).strip()
            parsed = json.loads(cleaned)
        else:
            parsed = raw

        for col in TARGET_COLUMNS:
            out_df.at[i, col] = str(parsed.get(col, "")).strip()

        print(f"Row {i + 1}/{len(out_df)} extracted in {round(time.time() - start_time, 2)}s")

    except Exception as exc:
        out_df.at[i, "Remarks"] = f"Extraction failed: {exc}"
        print(f"Row {i + 1}/{len(out_df)} failed: {exc}")

# -----------------------------
# Save to NEW CSV file (do not overwrite XLS)
# -----------------------------
out_df.to_csv(OUTPUT_FILE, index=False)
print(f"Done. Extracted CSV saved to: {OUTPUT_FILE.resolve()}")